In [6]:

"""
TASK 1: Data Immersion & Wrangling
ApexPlanet Software Pvt. Ltd. - Data Analytics Internship
Dataset: Indian Banking Transactions (550,000 rows)
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("TASK 1: Data Immersion & Wrangling")
print("=" * 60)

# ─── 1. LOAD DATA ────────────────────────────────────────────
df = pd.read_csv('indian_banking_transactions.csv')
print(f"\n[LOAD] Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ─── 2. DATA QUALITY ASSESSMENT ──────────────────────────────
print("\n--- DATA QUALITY ASSESSMENT ---")

# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print("\nMissing Values:")
print(pd.DataFrame({'Count': missing, 'Percentage': missing_pct})[missing > 0])

# Duplicates
dups = df.duplicated().sum()
print(f"\nDuplicate Rows: {dups}")

# Outlier detection (IQR method)
print("\nOutlier Detection (transaction_amount):")
Q1 = df['transaction_amount'].quantile(0.25)
Q3 = df['transaction_amount'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['transaction_amount'] < lower) | (df['transaction_amount'] > upper)]
print(f"  Outliers: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)")
print(f"  IQR bounds: [{lower:.2f}, {upper:.2f}]")

# Inconsistent categories
print("\nUnique values per categorical column:")
cats = ['account_type', 'transaction_type', 'transaction_direction',
        'transaction_status', 'channel', 'kyc_status', 'state']
for c in cats:
    print(f"  {c}: {sorted(df[c].unique().tolist())}")

# ─── 3. DATA CLEANING & TRANSFORMATION ───────────────────────
print("\n--- DATA CLEANING & TRANSFORMATION ---")

df_clean = df.copy()

# 3a. Standardize date format
df_clean['transaction_date'] = pd.to_datetime(df_clean['transaction_date'], format='%Y-%m-%d')
print(f"[DATE] transaction_date converted. Range: {df_clean['transaction_date'].min().date()} → {df_clean['transaction_date'].max().date()}")

# 3b. Fill missing loan_type
df_clean['loan_type'] = df_clean['loan_type'].fillna('No Loan')
print(f"[FILL] loan_type nulls filled with 'No Loan'. Remaining nulls: {df_clean['loan_type'].isnull().sum()}")

# 3c. Feature Engineering
# Year, Month, Day of week
df_clean['year']        = df_clean['transaction_date'].dt.year
df_clean['month']       = df_clean['transaction_date'].dt.month
df_clean['month_name']  = df_clean['transaction_date'].dt.strftime('%b')
df_clean['day_of_week'] = df_clean['transaction_date'].dt.day_name()
df_clean['quarter']     = df_clean['transaction_date'].dt.quarter

# Time-of-day bucket
def time_bucket(hour):
    if 6 <= hour < 12:   return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else:                 return 'Night'

df_clean['time_of_day'] = df_clean['transaction_hour'].apply(time_bucket)

# Credit score band
def score_band(s):
    if s >= 800:   return 'Excellent'
    elif s >= 700: return 'Good'
    elif s >= 600: return 'Fair'
    elif s >= 500: return 'Poor'
    else:          return 'Very Poor'

df_clean['credit_score_band'] = df_clean['credit_score'].apply(score_band)

# Amount category
def amount_cat(a):
    if a < 500:       return 'Micro (<500)'
    elif a < 5000:    return 'Small (500-5K)'
    elif a < 50000:   return 'Medium (5K-50K)'
    elif a < 500000:  return 'Large (50K-5L)'
    else:             return 'Very Large (>5L)'

df_clean['amount_category'] = df_clean['transaction_amount'].apply(amount_cat)

# Net transaction sign
df_clean['net_amount'] = df_clean.apply(
    lambda r: r['transaction_amount'] if r['transaction_direction'] == 'Credit'
              else -r['transaction_amount'], axis=1)

# is_weekend flag
df_clean['is_weekend'] = df_clean['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# is_high_value flag (top 5%)
threshold_95 = df_clean['transaction_amount'].quantile(0.95)
df_clean['is_high_value'] = (df_clean['transaction_amount'] >= threshold_95).astype(int)

print(f"\n[FEATURE] New columns added: year, month, month_name, day_of_week, quarter,")
print(f"          time_of_day, credit_score_band, amount_category, net_amount,")
print(f"          is_weekend, is_high_value")
print(f"[FEATURE] High-value threshold (95th pct): ₹{threshold_95:,.2f}")

# 3d. Outlier flag (don't remove, flag them)
df_clean['is_amount_outlier'] = (
    (df_clean['transaction_amount'] < lower) | (df_clean['transaction_amount'] > upper)
).astype(int)
print(f"[OUTLIER] Flagged {df_clean['is_amount_outlier'].sum():,} outliers (not removed)")

# ─── 4. FINAL SHAPE ──────────────────────────────────────────
print(f"\n[FINAL] Cleaned dataset shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"[FINAL] Total nulls remaining: {df_clean.isnull().sum().sum()}")

# ─── 5. SAVE ─────────────────────────────────────────────────
df_clean.to_csv('cleaned_dataset.csv', index=False)
print(f"\n[SAVED] cleaned_dataset.csv")

# ─── 6. DATA DICTIONARY ──────────────────────────────────────
data_dict = {
    'Column': [
        'transaction_id','customer_id','transaction_date','transaction_time',
        'account_type','transaction_type','transaction_amount','transaction_direction',
        'account_balance','merchant_category','state','credit_score','has_loan',
        'loan_type','emi_amount','transaction_status','channel','kyc_status',
        'is_fraud','transaction_hour',
        # Engineered
        'year','month','month_name','day_of_week','quarter','time_of_day',
        'credit_score_band','amount_category','net_amount','is_weekend',
        'is_high_value','is_amount_outlier'
    ],
    'Type': [
        'str','str','datetime','str','str','str','float','str','float','str',
        'str','int','int','str','float','str','str','str','int','int',
        'int','int','str','str','int','str','str','str','float','int','int','int'
    ],
    'Description': [
        'Unique transaction identifier',
        'Unique customer identifier',
        'Date of transaction (standardized to datetime)',
        'Time of transaction (HH:MM)',
        'Bank account type: Savings, Current, Salary, Fixed Deposit, NRI',
        'Payment method: UPI, NEFT, RTGS, ATM_Withdrawal, etc.',
        'Transaction amount in Indian Rupees (₹)',
        'Direction of money flow: Credit (in) or Debit (out)',
        'Account balance after transaction (₹)',
        'Merchant category for the transaction',
        'Indian state where transaction occurred',
        'Customer credit score (300-900 scale)',
        'Binary flag: 1=has active loan, 0=no loan',
        'Type of loan (Home/Personal/Auto/Gold/Education/No Loan)',
        'Monthly EMI amount in ₹; 0 if no loan',
        'Transaction outcome: Success, Failed, Pending',
        'Transaction channel: Mobile_App, Web, ATM, Branch, API, etc.',
        'KYC compliance status: Verified, Expired, Pending',
        'Fraud flag: 1=fraudulent, 0=legitimate',
        'Hour of day (0-23) when transaction occurred',
        '[ENGINEERED] Year extracted from transaction_date',
        '[ENGINEERED] Month number (1-12)',
        '[ENGINEERED] Month abbreviated name (Jan-Dec)',
        '[ENGINEERED] Day of the week',
        '[ENGINEERED] Calendar quarter (1-4)',
        '[ENGINEERED] Time bucket: Morning/Afternoon/Evening/Night',
        '[ENGINEERED] Credit score band: Excellent/Good/Fair/Poor/Very Poor',
        '[ENGINEERED] Transaction size category',
        '[ENGINEERED] Signed amount (positive=Credit, negative=Debit)',
        '[ENGINEERED] 1 if transaction on Saturday or Sunday',
        '[ENGINEERED] 1 if amount >= 95th percentile (₹{:.0f})'.format(threshold_95),
        '[ENGINEERED] 1 if amount is an IQR outlier'
    ],
    'Business_Relevance': [
        'Primary key for transaction lookup',
        'Customer segmentation & profiling',
        'Time-series trend analysis',
        'Intraday pattern detection',
        'Account-type behaviour comparison',
        'Payment channel preference analysis',
        'Revenue metrics & transaction sizing',
        'Cash-flow analysis',
        'Wealth segmentation',
        'Spending pattern analysis',
        'Geographic market analysis',
        'Credit risk assessment',
        'Loan portfolio analysis',
        'Loan product mix analysis',
        'EMI burden analysis',
        'Transaction success rate KPI',
        'Digital adoption metrics',
        'Compliance & risk monitoring',
        'Fraud rate KPI',
        'Peak usage hours analysis',
        'YoY growth analysis',
        'Monthly trend & seasonality',
        'Dashboard labelling',
        'Weekend vs weekday patterns',
        'Quarterly business reporting',
        'Day-part marketing & ops planning',
        'Customer creditworthiness segmentation',
        'Transaction volume tier analysis',
        'Net cash flow per transaction',
        'Weekend transaction behaviour',
        'High-value transaction monitoring',
        'Data quality flagging'
    ]
}

dd_df = pd.DataFrame(data_dict)
dd_df.to_csv('data_dictionary.csv', index=False)
print("[SAVED] data_dictionary.csv")
print("\n✅ Task 1 Complete!")


TASK 1: Data Immersion & Wrangling

[LOAD] Dataset loaded: 550,000 rows × 20 columns

--- DATA QUALITY ASSESSMENT ---

Missing Values:
            Count  Percentage
loan_type  377134       68.57

Duplicate Rows: 0

Outlier Detection (transaction_amount):
  Outliers: 88,976 (16.18%)
  IQR bounds: [-10928.65, 20249.19]

Unique values per categorical column:
  account_type: ['Current', 'Fixed Deposit', 'NRI', 'Salary', 'Savings']
  transaction_type: ['ATM_Withdrawal', 'Auto_Debit', 'Cheque', 'Credit_Card', 'IMPS', 'NEFT', 'Net_Banking', 'POS', 'RTGS', 'UPI']
  transaction_direction: ['Credit', 'Debit']
  transaction_status: ['Failed', 'Pending', 'Reversed', 'Success']
  channel: ['API', 'ATM', 'Branch', 'Mobile_App', 'POS_Terminal', 'Web']
  kyc_status: ['Expired', 'Pending', 'Verified']
  state: ['Delhi', 'Gujarat', 'Karnataka', 'Maharashtra', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Telangana', 'UP', 'West Bengal']

--- DATA CLEANING & TRANSFORMATION ---
[DATE] transaction_date converted. 